# Correctness-First LLM Router

Thin orchestrator over `src/`. Predicts expected per-model performance, routes by
`argmax(0.85*p_hat - 0.15*c_bar/denom + bias)` with OOF-tuned per-model bias.

Runs from **local Colab disk** (`/content/proj`) to insulate against Google Drive
FUSE drops; outputs + caches sync back to Drive after each phase, and are restored on
the next session (so the ~2 h encoder run resumes per-fold).

**If you change feature/model CODE**, delete `Output/router_v2/cache` on Drive so stale
predictions aren't reused. (Encoder caches are keyed by encoder config and are safe.)

In [ ]:
# Colab setup: mount Drive, copy project to LOCAL disk (insulates from Drive FUSE drops).
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import shutil, sys
from pathlib import Path
SRC = Path('/content/drive/MyDrive/NLP Final Project')
DST = Path('/content/proj'); DST.mkdir(exist_ok=True)

for sub in ['src', 'dataset']:
    dst = DST / sub
    if dst.exists(): shutil.rmtree(dst)
    shutil.copytree(SRC / sub, dst)

# Auto-discover cached Qwen3-Embedding-4B files anywhere under Output/ (run dirs vary).
emb_dst = DST / 'Output/router_a100/cache'; emb_dst.mkdir(parents=True, exist_ok=True)
found = []
for pat in ['qwen3_4b_dim1024_chunks_v1_*.npy', '*chunks*_train_*.npy', '*chunks*_test_*.npy']:
    found += list((SRC / 'Output').rglob(pat))
found = sorted(set(found))
for f in found: shutil.copy(f, emb_dst / f.name)
print('embedding cache files found+copied:', len(found))
for f in found: print('  from', f.relative_to(SRC / 'Output'))
if not found:
    print('--- no match; all .npy under Output/ (diagnosis): ---')
    for f in sorted((SRC / 'Output').rglob('*.npy'))[:60]: print('  ', f.relative_to(SRC / 'Output'))

# Restore prior router_v2 caches (Phase-0 learners + encoder per-fold checkpoints) for resume/speed.
v2 = SRC / 'Output/router_v2'
if v2.exists():
    shutil.copytree(v2, DST / 'Output/router_v2', dirs_exist_ok=True)
    n = len(list((DST / 'Output/router_v2/cache').glob('*'))) if (DST / 'Output/router_v2/cache').exists() else 0
    print(f'restored Output/router_v2 from Drive ({n} cache files)')

sys.path.insert(0, str(DST))

def sync_outputs():
    """Copy Output/router_v2 (submission + comparison + caches) back to Drive."""
    shutil.copytree(DST / 'Output/router_v2', SRC / 'Output/router_v2', dirs_exist_ok=True)
    print('synced Output/router_v2 -> Drive')

!pip -q install lightgbm 2>/dev/null
print('setup done; project at', DST)

## Phase 0 — reframe (TF-IDF + Qwen3 embeddings + handcrafted -> Ridge/LGBM -> calibrate -> bias-tuned routing)
Needs only `lightgbm` beyond Colab defaults.

In [ ]:
from src.config import CFG
from src.run import run_phase0
res0 = run_phase0(CFG(root=DST))   # add smoke=True for a fast dry run
sync_outputs()

## Phase 1 — ModernBERT-large encoder (multilabel + aux difficulty head)
GPU; ~2 h (5 folds + full-fit), checkpointed per fold. **Smoke-test first**, then full.
Reuses the Phase-0 classical learners (recomputed if their cache isn't present).

In [ ]:
!pip -q install 'transformers>=4.48.0' accelerate 2>/dev/null
from src.run import run_phase1
# Smoke first (~2-3 min) to validate the training loop end-to-end:
# res1 = run_phase1(CFG(root=DST, smoke=True, encoder_epochs=1, encoder_max_len=512, encoder_bs=4))
res1 = run_phase1(CFG(root=DST))   # full run; if OOM, set encoder_bs=4 or encoder_max_len=1024
sync_outputs()

## Phase 2 — vLLM self-consistency / judge difficulty features
Heavier install; GPU. (run_phase2 lands in Task 9.)

In [ ]:
!pip -q install vllm 2>/dev/null
from src.run import run_phase2
res2 = run_phase2(CFG(root=DST))
sync_outputs()

## Phase 3 — meta-stacker, bias refinement, final candidate submissions
(run_phase3 lands in Task 10.)

In [ ]:
from src.run import run_phase3
run_phase3(CFG(root=DST))
sync_outputs()